In [1]:
import pandas as pd
import numpy as np
import hashlib


c:\Users\TshushaP\AppData\Local\anaconda3\Lib\site-packages\pandas\core\computation\expressions.py:23: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.10.1' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED


In [2]:
def hash_student_id(student_no):
    if pd.isna(student_no):
        return None

    return hashlib.sha256(
        str(student_no).strip().encode("utf-8")
    ).hexdigest()

In [3]:
# import Academic record data and filter the years to include 2021 till current year
Acad = pd.read_excel("../../Academic/Academic records 2025-1.xlsx")

# Academic Data Exploration
print("Academic Data Shape:", Acad.shape)
print("Academic Data Info:\n", Acad.info())
print("Academic Data Missing Values:\n", Acad.isnull().sum())
print("Academic Data Columns Tolist:\n", Acad.columns.tolist())


Academic Data Shape: (983510, 19)
<class 'pandas.DataFrame'>
RangeIndex: 983510 entries, 0 to 983509
Data columns (total 19 columns):
 #   Column                     Non-Null Count   Dtype  
---  ------                     --------------   -----  
 0   Student_Number             983510 non-null  int64  
 1   Surname                    983510 non-null  str    
 2   Names                      983510 non-null  str    
 3   Year                       983510 non-null  int64  
 4   Qualification Code         983510 non-null  object 
 5   Qualification Description  983510 non-null  str    
 6   Qualification block        960419 non-null  object 
 7   Subject_Code               983510 non-null  str    
 8   Subject_Description        983510 non-null  str    
 9   Subject Block              983510 non-null  object 
 10  Subject Offering Type      983472 non-null  object 
 11  Offering Type Desc         983472 non-null  str    
 12  Period of Study            983492 non-null  object 
 13  Qualif

In [4]:
Acad.rename(columns={'Year': 'Academic Year'}, inplace=True)
Acad.rename(columns={'Student_Number': 'Student Number'}, inplace=True)

In [5]:
Acad["Identifier"] = Acad["Student Number"].apply(hash_student_id)

In [6]:
drop_cols= ["Student Number", "Surname", "Names"]
Acad.drop(columns= drop_cols, inplace=True)

In [7]:
Acad.head()

,Academic Year,Qualification Code,Qualification Description,Qualification block,Subject_Code,Subject_Description,Subject Block,Subject Offering Type,Offering Type Desc,Period of Study,Qualification Type,Qualification Type Name,Canceldate,Percentage,Result_Code,Result Description,Identifier
0,2025,DGCHER,PHD IN CHEMISTRY,2,CHE710R,RESEARCH PROJECT & DISSERTATION,2,2,PART-TIME: BELLVILLE CAMPUS,7,74,DOCTORAL DEGREE,NaN,NaN,NaN,NaN,ae684922102a68b0b2907a97e5050156c8ec229aed0179...
1,2026,DGCHER,PHD IN CHEMISTRY,2,CHE710R,RESEARCH PROJECT & DISSERTATION,0,2,PART-TIME: BELLVILLE CAMPUS,7,74,DOCTORAL DEGREE,NaN,NaN,NaN,NaN,ae684922102a68b0b2907a97e5050156c8ec229aed0179...
2,1983,208129,ND: MECHANICAL ENGINEERING,Q3,APT220T,APPLIED THERMODYNAMICS 2,2,NaN,NaN,2,23,NATIONAL DIPLOMA,NaN,59.0,P5,PASS,5a87b7b96e644f7580596b70c6d4c00423c0c458d3569a...
3,1983,208129,ND: MECHANICAL ENGINEERING,Q3,ESM110E,ENGINEERING SCIENCE (M) 1,1,NaN,NaN,1,23,NATIONAL DIPLOMA,NaN,60.0,P6,PASS,5a87b7b96e644f7580596b70c6d4c00423c0c458d3569a...
4,1983,208129,ND: MECHANICAL ENGINEERING,Q3,FLM220T,FLUID MECHANICS 2,2,NaN,NaN,2,23,NATIONAL DIPLOMA,NaN,43.0,FR,FAILED SUBMINIMUM,5a87b7b96e644f7580596b70c6d4c00423c0c458d3569a...


In [8]:
print("Academic Data Columns Tolist:\n", Acad.columns.tolist())

Academic Data Columns Tolist:
 ['Academic Year', 'Qualification Code', 'Qualification Description', 'Qualification block', 'Subject_Code', 'Subject_Description', 'Subject Block', 'Subject Offering Type', 'Offering Type Desc', 'Period of Study', 'Qualification Type', 'Qualification Type Name', 'Canceldate', 'Percentage', 'Result_Code', 'Result Description', 'Identifier']


In [9]:
# filter data to have years 2022 -2025, and Period of study 1, Qualification Description DIPLOMA IN ENG TECH:ELECTRICAL ENGINEER
Acad = Acad[
    (Acad['Period of Study'] == 1) & 
    (Acad['Qualification Description'] == 'DIPLOMA IN MECHANICAL ENGINEERING')]

Acad.drop(columns=['Result_Code','Result Description'], inplace=True)

In [10]:
# in ACAD I want to check the unique Subject_Description for each year, to know which subjects to pivot
Acad.groupby('Academic Year')['Subject_Description'].unique()

Academic Year
2018    [COMPUTER SKILLS AND APPLICATIONS, ENGINEERING...
2019    [COMPUTER SKILLS AND APPLICATIONS, ENGINEERING...
2020    [ENGINEERING MECHANICS, ENGINEERING MATHEMATIC...
2021    [PROFESSIONAL COMMUNICATION, COMPUTER SKILLS A...
2022    [COMPUTER SKILLS AND APPLICATIONS, ENGINEERING...
2023    [COMPUTER SKILLS AND APPLICATIONS, ENGINEERING...
2024    [ENGINEERING MATHEMATICS, PROFESSIONAL PRACTIC...
2025    [COMPUTER SKILLS AND APPLICATIONS, ENGINEERING...
2026    [ENGINEERING DRAWING 1, ENGINEERING MECHANICS,...
Name: Subject_Description, dtype: object

In [11]:
# In Acad change the Subject_Description values to be consistent across the years, for example in 2022 there is a subject called "Engineering Mathematics 1" and in 2023 there is a subject called "Engineering Mathematics 1A", I want to change the name of the subject in 2023 to be the same as in 2022, so that when I pivot the data, I will have one column for Engineering Mathematics 1 instead of two columns for Engineering Mathematics 1 and Engineering Mathematics 1A
Acad['Subject_Description'] = Acad['Subject_Description'].replace({
    'PROFESSIONAL PRACTICE AND COMMUNICATION': 'PROFESSIONAL COMMUNICATION'
})

In [12]:
# Step 1: explode the lists so each subject is its own row
df = Acad[['Academic Year', 'Subject_Description']].copy()

df = df.explode('Subject_Description')

# Step 2: create comparison matrix
module_matrix = pd.crosstab(
    df['Subject_Description'],
    df['Academic Year']
)

module_matrix

Academic Year,2018,2019,2020,2021,2022,2023,2024,2025,2026
Subject_Description,,,,,,,,,
COMPUTER SKILLS AND APPLICATIONS,21,51,71,145,164,166,200,327,30
ENGINEERING DRAWING 1,21,56,85,137,205,211,261,391,92
ENGINEERING MATHEMATICS,21,57,98,131,161,195,230,382,126
ENGINEERING MECHANICS,21,53,90,126,182,204,232,372,79
MECHANICAL ENGINEERING MANUFACTURING,21,54,78,119,157,171,204,370,30
PHYSICAL SCIENCE,21,48,72,120,148,208,198,347,46
PROFESSIONAL COMMUNICATION,21,49,75,131,181,170,214,329,26


In [13]:
# get unique student numbers per Year
Acad.groupby('Academic Year')['Identifier'].nunique()


Academic Year
2018     21
2019     62
2020    104
2021    170
2022    237
2023    267
2024    300
2025    476
2026    186
Name: Identifier, dtype: int64

In [14]:

# get unique Subject_Description for each year and keep first value of column Yearly_Result
Acad.groupby('Academic Year')['Subject_Description'].unique()
# pivot the data Subject_Description values to columns and thie correspondinf Percentage values to the new columns and add a column average marks and Yearly_Result
Acad_pivot = Acad.pivot_table(index=['Identifier','Academic Year'], columns='Subject_Description', values='Percentage', aggfunc='first').reset_index()


In [15]:
Acad_pivot.head(10)

Subject_Description,Identifier,Academic Year,COMPUTER SKILLS AND APPLICATIONS,ENGINEERING DRAWING 1,ENGINEERING MATHEMATICS,ENGINEERING MECHANICS,MECHANICAL ENGINEERING MANUFACTURING,PHYSICAL SCIENCE,PROFESSIONAL COMMUNICATION
0,00319f8c0080d567b39fbb3fd5ea0dc4e88c5901c47549...,2019,66.0,41.0,39.0,39.0,56.0,65.0,65.0
1,00319f8c0080d567b39fbb3fd5ea0dc4e88c5901c47549...,2020,NaN,69.0,78.0,89.0,NaN,NaN,NaN
2,0038486d0312881d3a8c2a7335b3a53346682cd2df88ff...,2025,56.0,27.0,50.0,43.0,51.0,54.0,NaN
3,006c8056517d18aaeada39e02766895a1caab6652ad014...,2021,70.0,65.0,37.0,51.0,59.0,69.0,55.0
4,006c8056517d18aaeada39e02766895a1caab6652ad014...,2022,NaN,NaN,54.0,NaN,NaN,NaN,NaN
5,00bbab96eb68f0bb38429c10a6e94670e05622419194ae...,2022,50.0,NaN,41.0,25.0,NaN,38.0,26.0
6,00d4a28e7d6909bdee2bc1cc9205f0f94e55028c637f71...,2025,69.0,64.0,26.0,56.0,65.0,50.0,70.0
7,00e0229a73e804258fdbaf0d08efa335a7b7b87a339803...,2023,63.0,55.0,50.0,56.0,76.0,60.0,75.0
8,0159be7baa1a0a80447a940984b30a3ca1c204b75f56d3...,2025,55.0,65.0,17.0,45.0,51.0,50.0,47.0
9,0159be7baa1a0a80447a940984b30a3ca1c204b75f56d3...,2023,NaN,12.0,NaN,3.0,NaN,NaN,NaN


In [16]:
Acad_pivot = Acad_pivot.copy()

# count non-null values per row (excluding ID columns)
Acad_pivot['non_null_count'] = Acad_pivot.drop(columns=['Identifier', 'Academic Year']).notna().sum(axis=1)

In [17]:
# Ensuring the dataset contains only the rows with the maximum non-null count for each student
Acad_pivot = Acad_pivot.loc[
    Acad_pivot.groupby('Identifier')['non_null_count'].idxmax()
]

Acad_pivot = Acad_pivot.drop(columns='non_null_count')

In [18]:
# from Acad create new column Average Marks which is the average of all the percentage columns for each student and year
percentage_columns = Acad_pivot.columns.difference(['Identifier', 'Academic Year'])
Acad_pivot['Average Marks'] = Acad_pivot[percentage_columns].mean(axis=1)


In [19]:
Acad_pivot.duplicated().sum()

np.int64(0)

In [20]:
Acad.to_excel("Academic records 2025-1.xlsx", index= False)

In [21]:
# Enrolment Data Exploration
Enrol = pd.read_excel("../../combined_enrolment_df.xlsx")
print("Enrolment Data Shape:", Enrol.shape)
print("Enrolment Data Info:\n", Enrol.info())
print("Enrolment Data Missing Values:\n", Enrol.isnull().sum())
print("Enrolment Data Columns Tolist:\n", Enrol.columns.tolist())

Enrolment Data Shape: (215578, 46)
<class 'pandas.DataFrame'>
RangeIndex: 215578 entries, 0 to 215577
Data columns (total 46 columns):
 #   Column                       Non-Null Count   Dtype         
---  ------                       --------------   -----         
 0   Academic Year                215578 non-null  int64         
 1   Faculty School               215578 non-null  int64         
 2   Faculty                      215578 non-null  str           
 3   Dept Code                    215578 non-null  int64         
 4   Department Name              215578 non-null  str           
 5   Student Number               215578 non-null  int64         
 6   Student Name                 215578 non-null  str           
 7   First Names                  215578 non-null  str           
 8   Surname                      215578 non-null  str           
 9   Gender                       215578 non-null  str           
 10  Home Language                215577 non-null  str           
 11  La

In [22]:
print("Enrolment Data Columns Tolist:\n", Enrol.columns.tolist())

Enrolment Data Columns Tolist:
 ['Academic Year', 'Faculty School', 'Faculty', 'Dept Code', 'Department Name', 'Student Number', 'Student Name', 'First Names', 'Surname', 'Gender', 'Home Language', 'Language Name', 'Ethnic Group Name', 'Citizenship Code', 'Country Name', 'Birthdate', 'Qualification Code', 'Qualification Name', 'Offering Type Code', 'Offering Type', 'Campus Code', 'Campus Name', 'Period Of Study', 'Ften Status This Year', 'Academic Block Code', 'Alien Indicator', 'Previous Activity', 'Previous Activity Desc', 'Id Number', 'Country Code', 'Marital Status', 'Passport Number', 'Source Of Funding', 'Matric Type', 'Secondary School', 'Secondary School Name', 'Created By Web', 'Subsidy Type', 'Qualification Type', 'Transaction Date', 'Post Graduate', 'Approved Qualification Code', 'Approved Qualification Name', 'Student Type', 'Student Type Description', 'School Quintile']


In [23]:
Enrol["Identifier"] = Enrol["Student Number"].apply(hash_student_id)

In [24]:
# Filter Enrolment to have years 2022 -2025, and Period Of Study 1, Qualification Description DIPLOMA IN ENG TECH:ELECTRICAL ENGINEER
Enrol = Enrol[ (Enrol['Academic Year'] >= 2022) & (Enrol['Academic Year'] <= 2025) &
    (Enrol['Period Of Study'] == 1) & 
    (Enrol['Qualification Name'] == 'DIPLOMA IN MECHANICAL ENGINEERING')&
    (Enrol["Ften Status This Year"]=="First Time Entering")]

In [25]:
Enrol.duplicated().sum()

np.int64(0)

In [26]:
Enrol.to_excel("combined_enrolment_df.xlsx",index = False)

In [27]:
Finance = pd.read_excel("../../FTE/Finance/Finance data.xlsx")
print("Finance Data Shape:", Finance.shape)
print("Finance Data Columns Tolist:\n", Finance.columns.tolist())
Finance.duplicated().sum()


Finance Data Shape: (563363, 11)
Finance Data Columns Tolist:
 ['StudentNo', 'Academic Year', 'Qualification Code', 'Qualification Name', 'Formal Time', 'Bursary Type', 'BURSARYCODE', 'BURSARYNAME', 'BURSARYFUND', 'FUNDNAME', 'Qualification Name.1']


np.int64(0)

In [28]:
Finance = Finance[ (Finance['Academic Year'] >= 2022) & (Finance['Academic Year'] <= 2025) &
    (Finance['Qualification Name'] == 'DIPLOMA IN MECHANICAL ENGINEERING')]

In [29]:
Finance.rename(columns={'StudentNo': 'Student Number'}, inplace=True)
Finance.columns.tolist()

['Student Number',
 'Academic Year',
 'Qualification Code',
 'Qualification Name',
 'Formal Time',
 'Bursary Type',
 'BURSARYCODE',
 'BURSARYNAME',
 'BURSARYFUND',
 'FUNDNAME',
 'Qualification Name.1']

In [30]:
Finance["Identifier"] = Finance["Student Number"].apply(hash_student_id)

In [31]:
Finance.drop(columns='Student Number', inplace = True)

In [32]:
Finance.to_excel("Finance data.xlsx", index= False)

In [33]:
Enrol = Enrol [['Academic Year', 'Faculty', 'Department Name', 'Identifier','Gender','Language Name', 'Ethnic Group Name', 'Country Name', 'Birthdate', 'Qualification Code', 'Qualification Name', 'Offering Type', 'Campus Name', 'Period Of Study', 'Ften Status This Year', 'Previous Activity Desc', 'Marital Status','Matric Type', 'Secondary School Name', 'Qualification Type','Post Graduate','Student Type Description', 'School Quintile']]



# left join Finance on Enrol by Academic Year and Student Number Drop Qualification Code and Qualification Name from Finance data
Enrol = Enrol.merge(Finance.drop(columns=[ 'Qualification Name.1','Qualification Code', 'Qualification Name','BURSARYCODE', 'BURSARYNAME', 'BURSARYFUND']), on=['Academic Year', 'Identifier'], how='left')
Enrol.columns.tolist()
Enrol.duplicated().sum()
Enrol.drop_duplicates(inplace=True)
Enrol.duplicated().sum()

np.int64(0)

In [34]:
# left join  Acad on Enrol by Academic Year and Student Number Drop Qualification from Acad data
Enrol = Enrol.merge(Acad_pivot, on=['Academic Year', 'Identifier'], how='inner')
Enrol.columns.tolist()

['Academic Year',
 'Faculty',
 'Department Name',
 'Identifier',
 'Gender',
 'Language Name',
 'Ethnic Group Name',
 'Country Name',
 'Birthdate',
 'Qualification Code',
 'Qualification Name',
 'Offering Type',
 'Campus Name',
 'Period Of Study',
 'Ften Status This Year',
 'Previous Activity Desc',
 'Marital Status',
 'Matric Type',
 'Secondary School Name',
 'Qualification Type',
 'Post Graduate',
 'Student Type Description',
 'School Quintile',
 'Formal Time',
 'Bursary Type',
 'FUNDNAME',
 'COMPUTER SKILLS AND APPLICATIONS',
 'ENGINEERING DRAWING 1',
 'ENGINEERING MATHEMATICS',
 'ENGINEERING MECHANICS',
 'MECHANICAL ENGINEERING MANUFACTURING',
 'PHYSICAL SCIENCE',
 'PROFESSIONAL COMMUNICATION',
 'Average Marks']

In [35]:
Enrol.duplicated().sum()

np.int64(0)

In [36]:
Res = pd.read_excel("../../FTE/Residence/Residence 2023_2026.xlsx")

res2 = pd.read_excel("../../FTE/Residence/residence_data2.xlsx")

Res = pd.concat([Res, res2], ignore_index=True)
# Residence Data Exploration
print("Residence Data Shape:", Res.shape)
print("Residence Data Info:\n", Res.info())
print("Residence Data Missing Values:\n", Res.isnull().sum())
print("Residence Data Columns Tolist:\n", Res.columns.tolist())

Residence Data Shape: (67817, 26)
<class 'pandas.DataFrame'>
RangeIndex: 67817 entries, 0 to 67816
Data columns (total 26 columns):
 #   Column                 Non-Null Count  Dtype         
---  ------                 --------------  -----         
 0   STUDENT                67817 non-null  int64         
 1   SURNAME                67817 non-null  str           
 2   FIRST_NAMES            67817 non-null  str           
 3   TITLE                  67817 non-null  str           
 4   GENDER                 67817 non-null  str           
 5   BIRTH_DATE             67817 non-null  datetime64[us]
 6   AGE                    67817 non-null  int64         
 7   SECONDARY_SCHOOL_CODE  66440 non-null  object        
 8   SECONDARY_SCHOOL_NAME  66440 non-null  str           
 9   ACADEMIC_YEAR          67817 non-null  int64         
 10  FACULTY                67817 non-null  int64         
 11  FACULTY_NAME           67817 non-null  str           
 12  DEPARTMENT             67817 non-null

In [37]:
# change Student to Student Number in Residence data
Res.rename(columns={'STUDENT': 'Student Number'}, inplace=True)
Res.rename(columns={'ACADEMIC_YEAR': 'Academic Year'}, inplace=True)

# create a new column Years_in_Residence that calculates the number of years a student was in residence by looking at the Academic Year and Residence Type
Res.columns.tolist()

['Student Number',
 'SURNAME',
 'FIRST_NAMES',
 'TITLE',
 'GENDER',
 'BIRTH_DATE',
 'AGE',
 'SECONDARY_SCHOOL_CODE',
 'SECONDARY_SCHOOL_NAME',
 'Academic Year',
 'FACULTY',
 'FACULTY_NAME',
 'DEPARTMENT',
 'DEPARTMENT_NAME',
 'QUALIFICATION',
 'PERIOD_STUDY',
 'OFFERING_TYPE',
 'BLOCK_CODE',
 'CAMPUS_CODE',
 'CAMPUS_NAME',
 'RESIDENCE_CODE',
 'RESIDENCE_NAME',
 'RESIDENCE_ROOM',
 'ROOM_TYPE',
 'RESIDENTIAL_ADRESS',
 'FTEN_STATUS']

In [38]:
Res["Identifier"] = Res["Student Number"].apply(hash_student_id)

In [39]:
# Create Yes/No column Res_status that indicates weather a student is in res or not for that year
Res['Res_status'] = np.where(Res['RESIDENCE_NAME'].notna(), 'Yes', 'No')

In [40]:
Res.to_excel("Residence 2023_2026.xlsx", index= False)

In [41]:
# leave Academic Year and Years_in_Residence in Residence data and drop the rest of the columns
Res = Res[['Academic Year', 'Identifier','Res_status','RESIDENCE_NAME']].copy()

# left join Residence on Enrol by Academic Year and Student Number
Enrol = Enrol.merge(Res, on=['Academic Year', 'Identifier'], how='left')

In [ ]:
grad = pd.read_excel("../../Graduation/Graduation data 2021-2025.xlsx")

# Grad Data Exploration
print("Graduation Data Shape:", grad.shape)
print("Graduation Data Info:\n", grad.info())
print("Graduation Data Missing Values:\n", grad.isnull().sum())
print("Graduation Data Columns Tolist:\n", grad.columns.tolist())

In [43]:
print(Enrol.head(5))
Enrol.columns.tolist()

   Academic Year                       Faculty  \
0           2022  ENGINEERING & BUILT ENVIRON.   
1           2022  ENGINEERING & BUILT ENVIRON.   
2           2025  ENGINEERING & BUILT ENVIRON.   
3           2022  ENGINEERING & BUILT ENVIRON.   
4           2024  ENGINEERING & BUILT ENVIRON.   

                  Department Name  \
0  MECHANICAL AND MECHATRONIC ENG   
1  MECHANICAL AND MECHATRONIC ENG   
2  MECHANICAL AND MECHATRONIC ENG   
3  MECHANICAL AND MECHATRONIC ENG   
4  MECHANICAL AND MECHATRONIC ENG   

                                          Identifier Gender Language Name  \
0  031cabecd64ce5caa03d22a2a271eadbbe2acc021a033f...      M       ENGLISH   
1  9adc4cfa1887f6ca47439a0dca8d068a5fed947a0e2820...      F       ENGLISH   
2  0c7a868d312a485253d66403713c6a55fbf68700d12930...      M       ENGLISH   
3  108b55b334d0e60ab104427cb03534360a20c200b18a9d...      M         XHOSA   
4  195ef62dd8161b41d9eddce1c990ec12d9186ce286552e...      M       ENGLISH   

  Ethnic Grou

['Academic Year',
 'Faculty',
 'Department Name',
 'Identifier',
 'Gender',
 'Language Name',
 'Ethnic Group Name',
 'Country Name',
 'Birthdate',
 'Qualification Code',
 'Qualification Name',
 'Offering Type',
 'Campus Name',
 'Period Of Study',
 'Ften Status This Year',
 'Previous Activity Desc',
 'Marital Status',
 'Matric Type',
 'Secondary School Name',
 'Qualification Type',
 'Post Graduate',
 'Student Type Description',
 'School Quintile',
 'Formal Time',
 'Bursary Type',
 'FUNDNAME',
 'COMPUTER SKILLS AND APPLICATIONS',
 'ENGINEERING DRAWING 1',
 'ENGINEERING MATHEMATICS',
 'ENGINEERING MECHANICS',
 'MECHANICAL ENGINEERING MANUFACTURING',
 'PHYSICAL SCIENCE',
 'PROFESSIONAL COMMUNICATION',
 'Average Marks',
 'Res_status',
 'RESIDENCE_NAME']

In [44]:
# ensure Birthdate column exists and is datetime
Enrol['Birthdate'] = pd.to_datetime(Enrol['Birthdate'], errors='coerce')

# compute age at registration (assuming registration date is Jan 1 of Academic Year)
birth = Enrol['Birthdate']
# subtract 1 if the birthday in that year occurs after Jan 1
not_had_birthday = ((birth.dt.month > 1) | ((birth.dt.month == 1) & (birth.dt.day > 1))).astype(int)
age_at_reg = Enrol['Academic Year'] - birth.dt.year - not_had_birthday

Enrol['Age_at_Registration'] = np.where(
    Enrol['Ften Status This Year'].str.lower() == 'first time entering',
    age_at_reg,
    np.nan
)


In [45]:
Enrol["Qualification Name"].unique()

<ArrowStringArray>
['DIPLOMA IN MECHANICAL ENGINEERING']
Length: 1, dtype: str

In [46]:
Enrol["Ften Status This Year"].unique()

<ArrowStringArray>
['First Time Entering']
Length: 1, dtype: str

In [47]:
Enrol["Academic Year"].unique()

array([2022, 2025, 2024, 2023])

In [48]:
Final_data = Enrol

In [49]:
# create a new column atrisk if in the percentage columns there is any value less than 50 the the answer is yes otherwise no from final data
percentage_columns = Acad_pivot.columns.difference(['Identifier', 'Academic Year', 'Average Marks'])
Final_data['At_Risk'] = np.where(Final_data[percentage_columns].lt(50).any(axis=1), 'Yes', 'No')

In [50]:
Final_data.columns.tolist()

['Academic Year',
 'Faculty',
 'Department Name',
 'Identifier',
 'Gender',
 'Language Name',
 'Ethnic Group Name',
 'Country Name',
 'Birthdate',
 'Qualification Code',
 'Qualification Name',
 'Offering Type',
 'Campus Name',
 'Period Of Study',
 'Ften Status This Year',
 'Previous Activity Desc',
 'Marital Status',
 'Matric Type',
 'Secondary School Name',
 'Qualification Type',
 'Post Graduate',
 'Student Type Description',
 'School Quintile',
 'Formal Time',
 'Bursary Type',
 'FUNDNAME',
 'COMPUTER SKILLS AND APPLICATIONS',
 'ENGINEERING DRAWING 1',
 'ENGINEERING MATHEMATICS',
 'ENGINEERING MECHANICS',
 'MECHANICAL ENGINEERING MANUFACTURING',
 'PHYSICAL SCIENCE',
 'PROFESSIONAL COMMUNICATION',
 'Average Marks',
 'Res_status',
 'RESIDENCE_NAME',
 'Age_at_Registration',
 'At_Risk']

In [51]:
Final_data["Academic Year"].unique()
Final_data = Final_data[
    (Final_data['Academic Year'] >= 2021) & (Final_data['Academic Year'] <= 2025) ]

In [52]:
Final_data.to_excel("../../Enrol_model.xlsx")